# 1. Đọc dữ liệu và kiểm tra phân bố tổng quan

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Tìm file dữ liệu
DATA_PATH = None
SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

for folder in SEARCH_ROOTS:
    candidates = [
        folder / "data" / "raw" / "RT_IOT2022.csv",
        folder / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
        folder / "final_IoT" / "sample_project" / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH is not None:
        break

if DATA_PATH is None:
    raise FileNotFoundError("Khong tim thay RT_IOT2022.csv trong cau truc project")

SAMPLE_ROOT = next(
    (
        candidate
        for folder in SEARCH_ROOTS
        for candidate in (folder, folder / "final_IoT" / "sample_project")
        if (candidate / "project").is_dir() and (candidate / "result").is_dir()
    ),
    None,
)
if SAMPLE_ROOT is None:
    raise FileNotFoundError("Khong tim thay thu muc goc sample_project")

RUN_DIR = SAMPLE_ROOT / "result" / "eda" / "EDA_Thanhdai" / "02_Data_Distribution"
FIG_DIR = RUN_DIR / "images"
TABLE_DIR = RUN_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

# Chỉ lấy các thuộc tính số
num = df.select_dtypes(include=np.number).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Loại cột hằng khỏi phép tính Skewness (không sửa df)
num_varying = num.loc[
    :, num.nunique(dropna=False) > 1
]

# Thống kê phân phối
summary = pd.DataFrame({
    "Min": num_varying.min(),
    "Median": num_varying.median(),
    "P99": num_varying.quantile(0.99),
    "Max": num_varying.max(),
    "Skewness": num_varying.skew(),
    "Zero (%)": num_varying.eq(0).mean() * 100
})

summary["Abs Skewness"] = summary["Skewness"].abs()

print("Kích thước dataset:", df.shape)

print(
    "Số thuộc tính có |Skewness| > 3:",
    (summary["Abs Skewness"] > 3).sum()
)

print(
    "Số thuộc tính có >= 95% giá trị 0:",
    (num.eq(0).mean() >= 0.95).sum()
)

display(
    summary.sort_values(
        "Abs Skewness",
        ascending=False
    ).head(15).round(3)
)

summary.to_csv(
    TABLE_DIR / "02_numeric_distribution_summary.csv",
    encoding="utf-8-sig"
)

### Nhận xét: Phân phối tổng quan

**Kết quả phân tích:**

- Phát hiện **65 đặc trưng số** có độ lệch tuyệt đối
  `|Skewness| > 3`.
- `bwd_bulk_bytes` có Skewness khoảng **340,50**.
- `active.std` có Skewness khoảng **287,99**.
- `bwd_pkts_tot` có Skewness khoảng **249,91**.
- Phát hiện **16 đặc trưng** có ít nhất 95%
  giá trị bằng 0, bao gồm cả cột hằng.

**Kết luận:**

Bộ dữ liệu có nhiều đặc trưng phân phối không đối xứng,
cần tiếp tục khảo sát bằng Histogram và các biểu đồ
phân phối khác.

# 2. Histogram: phát hiện phân phối lệch

In [ ]:
selected = [
    "flow_duration",
    "flow_iat.tot",
    "bwd_pkts_tot",
    "flow_pkts_per_sec",
    "bwd_bulk_bytes",
    "active.std"
]

selected = [c for c in selected if c in num.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, col in zip(axes.flat, selected):

    s = num[col].dropna()
    p99 = s.quantile(0.99)

    if p99 == 0:
        # Khi 99% dữ liệu bằng 0,
        # so sánh số mẫu bằng 0 và khác 0
        ax.bar(
            ["= 0", "!= 0"],
            [
                int((s == 0).sum()),
                int((s != 0).sum())
            ]
        )

        ax.set_title(f"{col} (P99 = 0)")

    else:
        # Chỉ giới hạn vùng hiển thị để dễ quan sát
        visible = s[s <= p99]

        ax.hist(
            visible.to_numpy(),
            bins=40
        )

        ax.set_title(f"{col} (hiển thị <= P99)")
        ax.set_xlabel(col)

    ax.set_ylabel("Số mẫu")

for ax in axes.flat[len(selected):]:
    ax.axis("off")

plt.suptitle(
    "EDA 02 - Histogram các đặc trưng",
    y=1.01
)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "02_histogram_distributions.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### Nhận xét: Histogram

**Kết quả phân tích:**

- `flow_duration` có trung vị rất nhỏ so với
  giá trị lớn nhất, khoảng 21.728,34.
- `bwd_pkts_tot` có trung vị bằng 1 nhưng
  giá trị lớn nhất lên đến 10.112.
- `bwd_bulk_bytes` có khoảng 99,10% giá trị bằng 0.
- `active.std` có khoảng 98,63% giá trị bằng 0.

**Kết luận:**

Các đặc trưng có phân phối lệch mạnh và nhiều
giá trị 0 cần được nghiên cứu sâu hơn.

# 3. So sánh phân phối gốc với Log1p

In [ ]:
features_log_view = [
    "flow_iat.tot",
    "bwd_pkts_tot",
    "bwd_bulk_bytes"
]

features_log_view = [
    c for c in features_log_view if c in num.columns
]

fig, axes = plt.subplots(
    len(features_log_view),
    2,
    figsize=(14, 11)
)

if len(features_log_view) == 1:
    axes = np.array([axes])

for row, col in enumerate(features_log_view):

    s = num[col].dropna()

    # log1p chỉ áp dụng cho các giá trị không âm
    assert (s >= 0).all(), (
        f"Cột {col} chứa giá trị âm, không phù hợp log1p"
    )

    # Histogram gốc
    axes[row, 0].hist(
        s.to_numpy(),
        bins=50
    )

    axes[row, 0].set_title(
        f"{col} - Phân phối gốc"
    )
    axes[row, 0].set_ylabel("Số mẫu")

    # Histogram sau khi đổi thang hiển thị
    axes[row, 1].hist(
        np.log1p(s.to_numpy()),
        bins=50
    )

    axes[row, 1].set_title(
        f"{col} - Trực quan hóa Log1p"
    )
    axes[row, 1].set_xlabel(
        f"log(1 + {col})"
    )
    axes[row, 1].set_ylabel("Số mẫu")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "02_original_vs_log1p_view.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### Nhận xét: So sánh phân phối gốc và Log1p

**Kết quả phân tích:**

- Ở thang đo gốc, các giá trị cực lớn kéo giãn
  trục ngang, khiến phần lớn mẫu tập trung
  gần giá trị 0 trên biểu đồ.
- Khi sử dụng thang Log1p, phạm vi giá trị được
  nén lại, giúp quan sát rõ hơn một số vùng
  phân bố của dữ liệu.
- Đối với các đặc trưng chứa nhiều số 0,
  Log1p không làm giảm tỷ lệ giá trị bằng 0.

**Kết luận:**

Các thuộc tính phân phối lệch phải có thể là
ứng viên để thử nghiệm Log Transformation.

Tuy nhiên, không phải mọi đặc trưng đều phù
hợp với phép biến đổi này.

# 4. KDE Plot: phân tích hình dạng phân phối

In [ ]:
from scipy.stats import gaussian_kde

col = "flow_iat.tot"

s = num[col].dropna()

# Chỉ lấy giá trị dương để khảo sát riêng
s_positive = s[s > 0]

# Lấy mẫu để vẽ nhanh, không thay đổi dữ liệu gốc
sample = s_positive.sample(
    n=min(20000, len(s_positive)),
    random_state=42
)

# Đổi thang hiển thị
view = np.log1p(sample)

fig, axes = plt.subplots(
    1, 2, figsize=(14, 4.5)
)

# Histogram
axes[0].hist(
    view.to_numpy(),
    bins=50,
    density=True
)

axes[0].set_title(
    "Histogram - log1p(flow_iat.tot)"
)
axes[0].set_xlabel("log1p(flow_iat.tot)")

# KDE
xs = np.linspace(
    view.min(),
    view.max(),
    400
)

kde = gaussian_kde(view.to_numpy())

axes[1].plot(
    xs,
    kde(xs)
)

axes[1].set_title(
    "KDE - log1p(flow_iat.tot)"
)
axes[1].set_xlabel("log1p(flow_iat.tot)")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "02_histogram_kde_flow_iat.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(
    f"Tỷ lệ {col} = 0:",
    f"{(s == 0).mean() * 100:.2f}%"
)

print(
    "Số mẫu dương dùng để vẽ:",
    len(sample)
)

### Nhận xét: KDE Plot

**Kết quả phân tích:**

- KDE giúp quan sát hình dạng phân phối và
  những vùng dữ liệu có mật độ cao.
- Biểu đồ được xây dựng từ các giá trị dương
  của `flow_iat.tot` sau khi chuyển thang Log1p.
- Các giá trị bằng 0 không được đưa vào KDE
  nên biểu đồ không đại diện cho toàn bộ
  phân phối ban đầu.

**Kết luận:**

Cần kết hợp KDE với Histogram, các phân vị
và tỷ lệ giá trị bằng 0 để đánh giá chính xác
hơn hình dạng phân phối.


# 5. Phân bố đặc trưng phân loại proto và service

In [ ]:
cat_cols = [
    c for c in ["proto", "service"]
    if c in df.columns
]

fig, axes = plt.subplots(
    1,
    len(cat_cols),
    figsize=(15, 5)
)

if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):

    counts = df[col].value_counts(dropna=False)

    table = pd.DataFrame({
        "Count": counts,
        "Percent (%)": (
            counts / len(df) * 100
        ).round(3)
    })

    print(f"\n=== Phân bố {col} ===")
    display(table)

    table.to_csv(
        TABLE_DIR / f"02_{col}_distribution.csv",
        encoding="utf-8-sig"
    )

    ax.barh(
        counts.index.astype(str),
        counts.values
    )

    ax.invert_yaxis()
    ax.set_title(f"Phân bố {col}")
    ax.set_xlabel("Số lượng mẫu")
    ax.set_ylabel(col)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "02_categorical_distribution.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### Nhận xét: Phân bố các đặc trưng phân loại

**1. Đặc trưng proto**

- `tcp` chiếm khoảng 89,69% tổng số mẫu.
- `udp` chiếm khoảng 10,26%.
- `icmp` chỉ chiếm khoảng 0,05%.

**2. Đặc trưng service**

- Giá trị `-` chiếm khoảng 83,55%.
- `dns` chiếm khoảng 7,92%.
- `mqtt` chiếm khoảng 3,36%.
- `http` chiếm khoảng 2,81%.

**Kết luận:**

Cần phân tích mối quan hệ giữa `proto`, `service`
và nhãn `Attack_type`.

Ký hiệu `-` chưa được xác định là dữ liệu thiếu,
do đó không được tự động thay thế hoặc xóa.